#Data Aggregation and Grouping Practice

#Practice Problems

In [ ]:
import pandas as pd

data = {
    "City": ["Lahore", "Karachi", "Lahore", "Islamabad", "Karachi", "Lahore", "Islamabad", "Karachi"],
    "Product": ["A", "B", "C", "A", "A", "B", "C", "C"],
    "Sales": [200, 300, 150, 400, 250, 350, 300, 200],
    "Profit": [50, 70, 40, 90, 60, 80, 75, 55]
}

df = pd.DataFrame(data)
df

,City,Product,Sales,Profit
0,Lahore,A,200,50
1,Karachi,B,300,70
2,Lahore,C,150,40
3,Islamabad,A,400,90
4,Karachi,A,250,60
5,Lahore,B,350,80
6,Islamabad,C,300,75
7,Karachi,C,200,55


#Problem 1: GroupBy Fundamentals + Basic Aggregation
####Find total Sales for each City


In [ ]:
df.groupby("City")["Sales"].sum()

,Sales
City,
Islamabad,700
Karachi,750
Lahore,700


#Problem 2: Multiple Aggregations
###For each City, calculate:
- total Sales
- average Profit
- maximum Sales


In [ ]:
df.groupby("City").agg({"Sales": ["sum","max"], "Profit":"mean"})

Sales          Profit
            sum  max       mean
City                           
Islamabad   700  400  82.500000
Karachi     750  300  61.666667
Lahore      700  350  56.666667

#Problem 3: Multi-Column Grouping
###Find total Sales for each City and Product combination


In [ ]:
df.groupby(["City","Product"])['Sales'].sum()

City       Product
Islamabad  A          400
           C          300
Karachi    A          250
           B          300
           C          200
Lahore     A          200
           B          350
           C          150
Name: Sales, dtype: int64

# Problem 4: Sorting Results
###Sort the result of Problem 3 by Sales in descending order


In [ ]:
df.groupby(["City","Product"])['Sales'].sum().sort_values(ascending=False)

City       Product
Islamabad  A          400
Lahore     B          350
Karachi    B          300
Islamabad  C          300
Karachi    A          250
           C          200
Lahore     A          200
           C          150
Name: Sales, dtype: int64

# Problem 5: Named Aggregations
####For each City, create:
- total_sales (sum of Sales)
- avg_profit (mean of Profit)


In [ ]:
df.groupby("City").agg(Total_Sales= ("Sales","sum"), Avg_Profit= ("Profit", "mean"))

,Total_Sales,Avg_Profit
City,,
Islamabad,700,82.500000
Karachi,750,61.666667
Lahore,700,56.666667


#Problem 6: Counting & Frequencies
1. Count number of records per City  
2. Count frequency of each Product  
3. Find Product frequency within each City


In [ ]:
df["City"].value_counts()
#or
df.groupby("City").size()

,0
City,
Islamabad,2
Karachi,3
Lahore,3


In [ ]:
df["Product"].value_counts()

,count
Product,
A,3
C,3
B,2


In [ ]:
df.groupby("Product").count()

,City,Sales,Profit
Product,,,
A,3,3,3
B,2,2,2
C,3,3,3


# Problem 7: Ranking
###Rank cities based on total Sales (highest = rank 1)


In [ ]:
df.groupby("City")['Sales'].sum().rank(method="dense", ascending= False)

,Sales
City,
Islamabad,2.0
Karachi,1.0
Lahore,2.0


#Problem 8: Ranking Within Groups
###Rank Products within each City based on Sales


In [ ]:
df.groupby("City")["Sales"] \
    .rank(method="dense", ascending=False)

,Sales
0,2.0
1,1.0
2,3.0
3,1.0
4,2.0
5,1.0
6,2.0
7,3.0


#Problem 9: Reset Index
###Convert a grouped result into a clean DataFrame (no MultiIndex)


In [ ]:
grouped = df.groupby(["City", "Product"])[["Sales","Profit"]].agg(["sum","mean"])
grouped

Sales        Profit      
                    sum   mean    sum  mean
City      Product                          
Islamabad A         400  400.0     90  90.0
          C         300  300.0     75  75.0
Karachi   A         250  250.0     60  60.0
          B         300  300.0     70  70.0
          C         200  200.0     55  55.0
Lahore    A         200  200.0     50  50.0
          B         350  350.0     80  80.0
          C         150  150.0     40  40.0

In [ ]:
grouped.reset_index()

City Product Sales        Profit      
                       sum   mean    sum  mean
0  Islamabad       A   400  400.0     90  90.0
1  Islamabad       C   300  300.0     75  75.0
2    Karachi       A   250  250.0     60  60.0
3    Karachi       B   300  300.0     70  70.0
4    Karachi       C   200  200.0     55  55.0
5     Lahore       A   200  200.0     50  50.0
6     Lahore       B   350  350.0     80  80.0
7     Lahore       C   150  150.0     40  40.0

#Problem 10: Group Filtering
###Keep only cities where total Sales > 700


In [ ]:
df.groupby("City").filter(lambda x: x["Sales"].sum() > 700).reset_index(drop=True)

,City,Product,Sales,Profit
0,Karachi,B,300,70
1,Karachi,A,250,60
2,Karachi,C,200,55


# Problem 11: Advanced Filtering (Professional Level)
###Keep only groups where:
- average Profit > 60

AND
- maximum Sales > 300
________________________________________


In [ ]:
df.groupby("City").filter(lambda x: (x["Profit"].mean() > 60) & (x["Sales"].max() > 300))

,City,Product,Sales,Profit
3,Islamabad,A,400,90
6,Islamabad,C,300,75


#Problem 12: Real-World Combined Problem
###Find top-performing City-Product combinations where:
- total Sales > 300
- sort by Sales descending
- rank them


In [ ]:
filtered= df.groupby(["City","Product"]).filter(lambda x: (x["Sales"].sum()> 300))
filtered

,City,Product,Sales,Profit
3,Islamabad,A,400,90
5,Lahore,B,350,80


Problem: I want only groups with Sales sum > 300

.filter() returns a flat DataFrame, not a grouped object

After above step, I only have rows for City–Product groups that meet the condition

In [ ]:
filtered["Product_rank"] = filtered.groupby("City")["Sales"].rank(method= "dense", ascending=False)
filtered["Product_rank"]

,Product_rank
3,1.0
5,1.0


Problem: I want to rank products within each City

.rank() must be applied on a Series (column), not the whole DataFrame

groupby("City") → ranks inside each city

method="dense" → no gaps in ranking

ascending=False → highest Sales = rank 1

In [ ]:
final_df = filtered.sort_values(by=["City","Product_rank"]).reset_index(drop=True)
final_df

,City,Product,Sales,Profit,Product_rank
0,Islamabad,A,400,90,1.0
1,Lahore,B,350,80,1.0


Problem: After filtering and ranking, the rows might be unordered

.sort_values() → makes it professional & readable

reset_index(drop=True) → cleans the row numbers